# B3 WSD — Taxonomy of LLM errors vs corpus-MFS (H6047)

Classifies **every** LLM miss on the frozen 300-item B3 set (`items.jsonl`, H5877:
LLM 54.3% vs corpus-MFS 84.3%, exact McNemar p<0.0001) into a mutually-exclusive
taxonomy with a verifiable sum, plus multi-label diagnostic tags.

Expected classes from the handoff: sandhi junctions, homoglossia, gloss deficit,
rare senses. The analysis below first verifies *what context the LLM actually saw*
and then assigns classes by deterministic precedence rules (nothing hand-typed).

Outputs (written beside this notebook):
- `error_taxonomy.tsv` — one row per LLM miss, primary class + soft tags
- `taxonomy_summary.json` — counts, shares, sum-check, parameters

Run with the DCS database available (`KOSHA_DCS_SQLITE` env override, else the
`wsd_core.DEFAULT_DCS` path, else the main-clone fallback); stdlib only.

In [ ]:
import json, csv, re, os, sqlite3, sys, collections

def find_repo_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.exists(os.path.join(d, "scripts", "wsd_iscls9_eval.py")):
            return d
        d = os.path.dirname(d)
    raise SystemExit("repo root not found (run from inside the kosha clone)")

ROOT = find_repo_root(os.getcwd())
EVAL = os.path.join(ROOT, "data", "eval", "wsd_iscls9")
sys.path.insert(0, os.path.join(ROOT, "scripts"))
print("repo root:", ROOT)

repo root: /Users/mac/Documents/GitHub-worktrees/kosha/kosha-h6047-drain


In [ ]:
def resolve_dcs():
    cands = []
    if os.environ.get("KOSHA_DCS_SQLITE"):
        cands.append(os.environ["KOSHA_DCS_SQLITE"])
    try:
        from wsd_core import DEFAULT_DCS
        cands.append(DEFAULT_DCS)
    except Exception:
        pass
    cands.append(os.path.normpath(os.path.join(ROOT, "..", "..", "..", "GitHub",
                                               "VisualDCS", "src", "DCS-data-2026",
                                               "dcs_full.sqlite")))
    for c in cands:
        if c and os.path.exists(c):
            return c
    return None

DCS = resolve_dcs()
print("DCS db:", DCS if DCS else "UNAVAILABLE (context-verification cells will report n/a)")

DCS db: /Users/mac/Documents/GitHub/VisualDCS/src/DCS-data-2026/dcs_full.sqlite


In [ ]:
items = [json.loads(l) for l in open(os.path.join(EVAL, "items.jsonl"), encoding="utf-8")]
scores = json.load(open(os.path.join(EVAL, "scores.json"), encoding="utf-8"))
per = list(csv.DictReader(open(os.path.join(EVAL, "per_item.tsv"), encoding="utf-8"),
                          delimiter="\t"))
per_by = {(r["sentence_id"], r["lemma_slp1"]): r for r in per}
by_id = {f"{it['sentence_id']}|{it['lemma_slp1']}": it for it in items}
errors = [r for r in per if r["llm_right"] == "0"]
print(f"items={len(items)} per_item={len(per)} LLM misses={len(errors)} "
      f"(accuracy {scores['llm']['accuracy']}, MFS {scores['mfs']['accuracy_on_scored']})")
assert len(errors) == round(len(items) * (1 - scores["llm"]["accuracy"]))

items=300 per_item=300 LLM misses=137 (accuracy 0.5433, MFS 0.8433)


## 1. Context verification — which sentence did the LLM actually see?

`scan_join` (scripts/wsd_iscls9_eval.py) records `token.sentence_id` (internal
INTEGER key) as the item sentence id, but `cmd_build` attaches context text via
`SELECT text_sandhied FROM sentence WHERE sent_id = ?` — the **external TEXT** id.
If those two key spaces differ, every item carries the *wrong* sentence. Verify
against the DB (exact `lemma_id` join, no string heuristics):

In [ ]:
lid2slp = None
if DCS:
    from wsd_core import load_lemma_map
    con = sqlite3.connect(f"file:{DCS}?mode=ro", uri=True)
    lid2slp = load_lemma_map(DCS)
    slp2lids = {}
    for lid, slp in lid2slp.items():
        slp2lids.setdefault(slp, set()).add(lid)
    n = true_has = shown_has = shown_eq = 0
    for it in items:
        sid = int(it["sentence_id"]); lids = slp2lids.get(it["lemma_slp1"], set())
        inlist = ",".join("?" * len(lids))
        t = con.execute(f"SELECT 1 FROM token WHERE sentence_id=? AND lemma_id IN ({inlist}) LIMIT 1",
                        (sid, *lids)).fetchone()
        srow = con.execute("SELECT id, text_sandhied FROM sentence WHERE sent_id=?",
                           (str(sid),)).fetchone()
        s = con.execute(f"SELECT 1 FROM token WHERE sent_id=? AND lemma_id IN ({inlist}) LIMIT 1",
                        (str(sid), *lids)).fetchone() if srow else None
        truetext = con.execute("SELECT text_sandhied FROM sentence WHERE id=?", (sid,)).fetchone()
        n += 1; true_has += bool(t); shown_has += bool(s)
        shown_eq += bool(srow and truetext and srow[1] == truetext[0])
        it["_shown_has_target"] = bool(s)
        it["_true_has_target"] = bool(t)
    print(f"items={n}  TRUE sentence (sentence.id=token.sentence_id) contains target: {true_has}")
    print(f"items={n}  SHOWN sentence (sentence.sent_id=<internal int>) contains target: {shown_has}")
    print(f"shown text == true text: {shown_eq}")
else:
    print("DCS db unavailable — re-run with the database to reproduce context verification")

items=300  TRUE sentence (sentence.id=token.sentence_id) contains target: 300
items=300  SHOWN sentence (sentence.sent_id=<internal int>) contains target: 2
shown text == true text: 0


## 2. Feature computation

Glosses stored in `items.jsonl` are exactly what the model was shown (the prompt
re-truncates at 220 chars; stored glosses are already shorter). Operational rules:

- **homoglossia**: content-word Jaccard(gold gloss, picked gloss) ≥ 0.5
  (content words: lowercase alpha tokens ≥3 chars minus a stoplist) — the choice
  is lexically near-arbitrary because the two glosses overlap heavily.
- **gloss_deficit**: the gold gloss as shown is uninformative — fewer than 3
  content words, or under 40 chars, or truncated at the storage cap without
  terminal punctuation.
- **rare_sense**: gold sense rank ≥ 4 in the attested inventory, or its corpus
  attestation share < 10% (`data/frequency/sense_frequency.tsv`, mw layer).
- **sandhi_fusion**: the target's surface form is not a standalone
  whitespace-delimited word of the sentence the LLM saw (fused via sandhi or
  compounding) — only computable when the target is present at all.
- **prior_bias** (soft only): the model picked a rank-1/2 "prototypical" sense
  while gold ranks lower — the signature of a context-free policy.

In [ ]:
STOP = {"the","a","an","of","and","or","in","to","with","esp","cf","qv","andc","etc",
         "see","q.v","viz","i.e","e.g","not","any","one","also","which","who","that"}
def words(g):
    return [w for w in re.findall(r"[a-z\u0100-\u017f]{3,}", (g or "").lower()) if w not in STOP]
def jaccard(a, b):
    A, B = set(words(a)), set(words(b))
    return len(A & B) / len(A | B) if (A or B) else 0.0

# corpus attestation share per sense (mw attested layer)
share = {}
try:
    with open(os.path.join(ROOT, "data", "frequency", "sense_frequency.tsv"), encoding="utf-8") as f:
        for r in csv.DictReader(f, delimiter="\t"):
            if r.get("layer") != "mw" or (r.get("provenance") or "attested") != "attested":
                continue
            try: c = float(r.get("lemma_share") or 0.0)
            except ValueError: c = 0.0
            share[(r["lemma_slp1"], r["sense_id"])] = c
except FileNotFoundError:
    share = {}
    print("sense_frequency.tsv not found — share-based rare_sense rule disabled")

GLOSS_CAP = max((len(c["gloss"]) for it in items for c in it["candidates"]), default=0)
print("stored gloss length cap observed:", GLOSS_CAP)

rows = []
for r in errors:
    it = by_id[f"{r['sentence_id']}|{r['lemma_slp1']}"]
    cands = {c["sense_id"]: c for c in it["candidates"]}
    gold, pick = it["gold"], r["llm_pick"]
    gg, pg = cands[gold]["gloss"], cands[pick]["gloss"]
    grank, prank = cands[gold]["rank"], cands[pick]["rank"]
    gw = words(gg)
    share_g = share.get((it["lemma_slp1"], gold), 0.0)
    fused = None
    if DCS and it.get("_shown_has_target"):
        lids = slp2lids.get(it["lemma_slp1"], set())
        shown = con.execute("SELECT text_sandhied FROM sentence WHERE sent_id=?",
                            (it["sentence_id"],)).fetchone()[0]
        forms = [f for (f,) in con.execute(
            f"SELECT DISTINCT form FROM token WHERE sent_id=? AND lemma_id IN ({','.join('?'*len(lids))})",
            (it["sentence_id"], *lids))]
        fused = not any(f in shown.split() for f in forms)
    rows.append({
        "sentence_id": r["sentence_id"], "lemma_slp1": it["lemma_slp1"], "gold": gold,
        "llm_pick": pick, "gold_rank": grank, "pick_rank": prank, "k": r["k"],
        "mfs_pred": r["mfs_pred"], "mfs_right": r["mfs_right"],
        "shown_has_target": int(it.get("_shown_has_target", False)),
        "true_has_target": int(it.get("_true_has_target", False)),
        "homoglossia": int(jaccard(gg, pg) >= 0.5),
        "gloss_deficit": int(len(gw) < 3 or len(gg) < 40 or
                             (len(gg) >= GLOSS_CAP and gg[-1:] not in ".;)!?")),
        "rare_sense": int(grank >= 4 or share_g < 0.10),
        "gold_top1_missed": int(grank == 1 and prank >= 2),
        "sandhi_fusion_shown": "" if fused is None else int(fused),
        "prior_bias": int(prank <= 2 and grank > prank),
        "gloss_jaccard": round(jaccard(gg, pg), 3),
        "gold_share": round(share_g, 4),
        "gold_gloss": gg, "pick_gloss": pg,
    })
print("features computed for", len(rows), "misses")

stored gloss length cap observed: 100
features computed for 137 misses


## 3. Primary class assignment (mutually exclusive, precedence, verifiable sum)

Precedence — first matching rule wins, every miss gets exactly one class:

1. `ctx_mismatch` — target lemma absent from the sentence the LLM was shown
2. `sandhi_fusion` — target present but surface form fused (not a standalone word)
3. `homoglossia` — gold vs picked gloss overlap ≥ 0.5
4. `gloss_deficit` — gold gloss uninformative as shown
5. `rare_sense` — gold rank ≥ 4 or corpus share < 10%
6. `context_ambiguous` — residual: real context-based discrimination failure

In [ ]:
def primary(x):
    if DCS is not None:
        if not x["shown_has_target"]: return "ctx_mismatch"
        if x["sandhi_fusion_shown"] == 1: return "sandhi_fusion"
    if x["homoglossia"]: return "homoglossia"
    if x["gloss_deficit"]: return "gloss_deficit"
    if x["rare_sense"]: return "rare_sense"
    return "context_ambiguous"

for x in rows:
    x["primary_class"] = primary(x)

cnt = collections.Counter(x["primary_class"] for x in rows)
n = len(rows)
print(f"{'class':<20} {'n':>4} {'share':>7}")
for cls in ["ctx_mismatch", "sandhi_fusion", "homoglossia", "gloss_deficit",
            "rare_sense", "context_ambiguous"]:
    if cnt.get(cls):
        print(f"{cls:<20} {cnt[cls]:>4} {cnt[cls]/n:>7.1%}")
assert sum(cnt.values()) == n == 137, "sum check failed"
print(f"SUM CHECK: {sum(cnt.values())}/{n} — 100% of misses classified")

class                   n   share
ctx_mismatch          136   99.3%
sandhi_fusion           1    0.7%
SUM CHECK: 137/137 — 100% of misses classified


In [ ]:
soft = {k: sum(x[k] for x in rows if x[k] != "") for k in
         ["homoglossia", "gloss_deficit", "rare_sense", "prior_bias", "gold_top1_missed"]}
soft["sandhi_fusion(where computable)"] = sum(1 for x in rows if x["sandhi_fusion_shown"] == 1)
jac_vals = sorted(x["gloss_jaccard"] for x in rows)
print(f"gloss jaccard sensitivity: >=0.2: {sum(v >= 0.2 for v in jac_vals)}  "
      f">=0.3: {sum(v >= 0.3 for v in jac_vals)}  median: {jac_vals[len(jac_vals)//2]}")
print(f"gold sense corpus rank among misses: rank1={sum(1 for x in rows if x['gold_rank']==1)} "
      f"rank2={sum(1 for x in rows if x['gold_rank']==2)} rank3+={sum(1 for x in rows if x['gold_rank']>=3)}")
print("Soft diagnostic tags over ALL misses (multi-label):")
for k, v in soft.items():
    print(f"  {k:<34} {v:>4}  ({v/n:.1%})")
mfs_right_on_miss = sum(1 for x in rows if x["mfs_right"] == "1")
print(f"\nMFS was right on {mfs_right_on_miss}/{n} of the LLM's misses "
      f"(the rest both wrong: {n - mfs_right_on_miss})")

examples = {}
for cls in cnt:
    examples[cls] = [
        {"item": f"{x['sentence_id']}|{x['lemma_slp1']}", "gold": x["gold"],
         "picked": x["llm_pick"], "gold_gloss": x["gold_gloss"][:90],
         "picked_gloss": x["pick_gloss"][:90]}
        for x in rows if x["primary_class"] == cls][:2]
print(json.dumps(examples["ctx_mismatch"][0], indent=1) if "ctx_mismatch" in examples else "")

gloss jaccard sensitivity: >=0.2: 7  >=0.3: 0  median: 0.071
gold sense corpus rank among misses: rank1=102 rank2=17 rank3+=18
Soft diagnostic tags over ALL misses (multi-label):
  homoglossia                           0  (0.0%)
  gloss_deficit                        61  (44.5%)
  rare_sense                           19  (13.9%)
  prior_bias                           30  (21.9%)
  gold_top1_missed                    100  (73.0%)
  sandhi_fusion(where computable)       1  (0.7%)

MFS was right on 100/137 of the LLM's misses (the rest both wrong: 37)
{
 "item": "101437|pitf",
 "gold": "pitf#2",
 "picked": "pitf#3",
 "gold_gloss": "pitf/ m. du. ( \u02datarO ) father and mother, parents, RV. &c. &c. (in the Veda N. of the Ara\u1e47",
 "picked_gloss": "( \u02dataras ) the fathers, forefathers, ancestors, ( esp. ) the Pit\u1e5b s or deceased ancestors "
}


In [ ]:
tsv = os.path.join(EVAL, "error_taxonomy.tsv")
with open(tsv, "w", encoding="utf-8", newline="\n") as f:
    w = csv.DictWriter(f, fieldnames=[k for k in rows[0] if k not in ("gold_gloss", "pick_gloss")]
                       + ["gold_gloss", "pick_gloss"], delimiter="\t")
    w.writeheader(); w.writerows(rows)

summary = {
    "handoff": "H6047",
    "input": {"items": "items.jsonl (H5877)", "n_items": len(items),
              "llm_accuracy": scores["llm"]["accuracy"],
              "mfs_accuracy": scores["mfs"]["accuracy_on_scored"]},
    "n_llm_misses": n,
    "context_verification": {
        "scope": "all items",
        "true_sentence_has_target": sum(bool(it.get("_true_has_target")) for it in items),
        "shown_sentence_has_target": sum(bool(it.get("_shown_has_target")) for it in items),
        "verdict": "sentence-context join bug: item sentence_id is token.sentence_id "
                   "(internal int) but text attached via sentence.sent_id (external text) "
                   "— scripts/wsd_iscls9_eval.py scan_join/cmd_build; fix: attach text by "
                   "sentence.id (=token.sentence_id) or key both sides on sent_id",
    } if DCS else {"verdict": "DB unavailable at generation time"},
    "primary_class_counts": dict(cnt),
    "primary_class_shares": {k: round(v / n, 4) for k, v in cnt.items()},
    "soft_tag_counts": soft,
    "sum_check": {"classified": sum(cnt.values()), "total": n, "ok": sum(cnt.values()) == n},
    "parameters": {"homoglossia_jaccard": 0.5, "gloss_deficit_maxcap": GLOSS_CAP,
                   "rare_sense_rank": 4, "rare_sense_share": 0.10,
                   "share_column": "sense_frequency.tsv:lemma_share (mw/attested)",
                   "sensitivity": {"jaccard_ge_0.2": sum(v >= 0.2 for v in jac_vals),
                                   "jaccard_ge_0.3": sum(v >= 0.3 for v in jac_vals)},
                   "gold_rank_distribution": dict(collections.Counter(x["gold_rank"] for x in rows))},
}
with open(os.path.join(EVAL, "taxonomy_summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("wrote", tsv)
print("wrote", os.path.join(EVAL, "taxonomy_summary.json"))

wrote /Users/mac/Documents/GitHub-worktrees/kosha/kosha-h6047-drain/data/eval/wsd_iscls9/error_taxonomy.tsv
wrote /Users/mac/Documents/GitHub-worktrees/kosha/kosha-h6047-drain/data/eval/wsd_iscls9/taxonomy_summary.json


## 4. Findings

1. **Dominant class = `ctx_mismatch`** (see §1): the LLM was shown the *wrong*
   sentence in ~99% of items — the context join keys two different id spaces.
   The 54.3% therefore measures a **context-free** gloss-prior policy, not
   in-context WSD. Corpus-MFS (84.3%) and the floor (31.9%) are unaffected
   (they never read sentence text).
2. Secondary soft tags quantify how the context-free policy misses: prior/prototypical
   bias, homoglossia, gloss deficit, rare senses — each with the deterministic rule
   recorded in `taxonomy_summary.json → parameters`.
3. Consequences: any MFS+LLM hybrid (H6048) must first land the one-line context-join
   fix and re-run the LLM arm; the ISCLS-9 B3 paper's LLM-vs-MFS negative needs
   re-measurement before submission. See
   `docs/ISCLS9_B3_WSD_ERROR_TAXONOMY.md` for the full report.